## 1. Install TPU-Compatible JAX Stack
This notebook is rebuilt as a clean, ordered flow. We begin by installing a TPU-safe JAX stack appropriate for Kaggle TPU v3-8. We use JAX 0.4.34 TPU wheels and align NumPy and ml-dtypes. We defer MaxText deps to after clone.

Key goals:
- Ensure 8 TPU devices are accessible
- Avoid resolver upgrades that break TPU wheels


In [1]:
# 1) Install TPU-safe JAX stack
!pip install --no-deps --force-reinstall \
  "numpy==1.26.4" \
  "ml-dtypes==0.4.0" \
  --quiet
!pip install --no-deps --force-reinstall \
  "jaxlib==0.4.34" \
  --quiet
!pip install --no-deps --force-reinstall \
  "jax[tpu]==0.4.34" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html \
  --quiet

import jax
print("JAX:", jax.__version__, "TPU devices:", jax.device_count())



[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


E0000 00:00:1757433171.485152      10 common_lib.cc:612] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: === 
learning/45eac/tfrc/runtime/common_lib.cc:230


JAX: 0.4.34 TPU devices: 8


## 2. Clone MaxText at a Compatible Commit
Clone the MaxText repository and pin to a pre-pallas commit that does not require `jax.experimental.pallas.ops.attention`. We first try a Git-based search; if inconclusive, we fall back to manual scanning of recent commits.


In [2]:
# 2) Clone and pin MaxText
!git clone https://github.com/google/maxtext.git || true
%cd /kaggle/working/maxtext

import subprocess, os

# Try to find introduction commit for pallas.ops.attention and checkout its parent
patterns = [
    "pallas.ops.attention",
    "from jax.experimental.pallas.ops import attention",
]
culprit = None
for pattern in patterns:
    r = subprocess.run(['git', 'log', '-S', pattern, '--pretty=format:%H', '-n', '1'], capture_output=True, text=True)
    if r.returncode == 0 and r.stdout.strip():
        culprit = r.stdout.strip().split('\n')[0]
        break

if culprit:
    print("First commit with pallas.ops.attention:", culprit)
    subprocess.run(['git', 'checkout', f'{culprit}^'], check=False)
    print(subprocess.check_output(['git', 'show', '-s', '--format=%ci %H', 'HEAD'], text=True))
else:
    # Fallback: pick a known pre-pallas date range (e.g., <= 2024-03-15) and pick that commit
    fallback = subprocess.check_output(['git', 'rev-list', '-n', '1', '--before=2024-03-15', 'HEAD'], text=True).strip()
    if fallback:
        print("Fallback commit (pre-2024-03-15):", fallback)
        subprocess.run(['git', 'checkout', fallback], check=False)
        print(subprocess.check_output(['git', 'show', '-s', '--format=%ci %H', 'HEAD'], text=True))
    else:
        print("Warning: could not determine a pre-pallas commit; staying on current HEAD.")

# Sanity: detect pallas import in current tree
has_pallas = False
try:
    with open('MaxText/layers/attentions.py', 'r') as f:
        has_pallas = 'pallas.ops.attention' in f.read()
except FileNotFoundError:
    pass
print("attentions.py uses pallas:", has_pallas)


/usr/local/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


Cloning into 'maxtext'...
remote: Enumerating objects: 55505, done.
remote: Counting objects: 100% (1228/1228), done.
remote: Compressing objects: 100% (591/591), done.
remote: Total 55505 (delta 960), reused 652 (delta 632), pack-reused 54277 (from 2)
Receiving objects: 100% (55505/55505), 317.28 MiB | 29.53 MiB/s, done.
Resolving deltas: 100% (40887/40887), done.


/usr/local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/kaggle/working/maxtext
First commit with pallas.ops.attention: 2a6154f254bf5dbe67e659360775a83a797ed7f9


Note: switching to '2a6154f254bf5dbe67e659360775a83a797ed7f9^'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at a77e867b Merge pull request #682 from google:anfals/up_gpu_base_image


2024-06-04 09:32:27 -0700 a77e867b27aff9d800e344b2538869c636d0f0f3

attentions.py uses pallas: False


## 2b. Auto-Rollback to First Pre-Pallas Commit
The previous attempt still hit a `pallas.ops.attention` import. This helper walks back through history to the earliest commit where `MaxText/layers/attentions.py` does not reference `pallas.ops.attention`, then stops there.


## 2b. Auto-Rollback to First Pre-Pallas Commit
The previous attempt still hit a `pallas.ops.attention` import. This helper walks back through history to the earliest commit where `MaxText/layers/attentions.py` does not reference `pallas.ops.attention`, then stops there.


In [ ]:
# 2b) Walk back until pallas import disappears
%cd /kaggle/working/maxtext

import subprocess, os

# enumerate many commits back
log = subprocess.check_output(['git', 'log', '--pretty=%H', '-n', '200'], text=True).strip().split('\n')
print("Scanning", len(log), "commits...")

chosen = None
for idx, h in enumerate(reversed(log)):  # oldest -> newest
    subprocess.run(['git', 'checkout', h], check=True)
    try:
        with open('MaxText/layers/attentions.py', 'r') as f:
            content = f.read()
    except FileNotFoundError:
        continue
    if 'pallas.ops.attention' not in content:
        chosen = h
        print("Found pre-pallas commit:", h)
        break

if chosen is None:
    print("Did not find a pre-pallas commit in scanned window. Staying on current.")
else:
    print(subprocess.check_output(['git', 'show', '-s', '--format=%ci %H', 'HEAD'], text=True))

# Show status
try:
    with open('MaxText/layers/attentions.py', 'r') as f:
        print("pallas present:", 'pallas.ops.attention' in f.read())
except FileNotFoundError:
    print("attentions.py missing")


## 3. Install MaxText Dependencies (Avoid Upgrading JAX)
Install MaxText requirements but protect the JAX pins by reapplying them immediately after. This balances repo requirements with TPU-safe versions.


In [3]:
# 3) Install MaxText deps, then re-pin JAX stack
%cd /kaggle/working/maxtext
!pip install -r requirements.txt --quiet || true

# Re-assert JAX pins to prevent resolver upgrades breaking TPU wheels
!pip install --no-deps --force-reinstall \
  "numpy==1.26.4" \
  "ml-dtypes==0.4.0" \
  --quiet
!pip install --no-deps --force-reinstall \
  "jaxlib==0.4.34" \
  --quiet
!pip install --no-deps --force-reinstall \
  "jax[tpu]==0.4.34" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html \
  --quiet
!pip install --no-deps --force-reinstall \
  "flax==0.10.4" \
  "optax==0.2.5" \
  "chex==0.1.89" \
  "orbax-checkpoint==0.11.5" \
  --quiet

import jax, flax, optax
print("JAX:", jax.__version__, "Flax:", flax.__version__, "Optax:", optax.__version__)


/usr/local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/kaggle/working/maxtext


/usr/local/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()



[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
JAX: 0.4.34 Flax: 0.10.4 Optax: 0.2.5


## 4. Verify TPU Devices and Basic JAX Runtime
Quick sanity check: confirm 8 TPU devices and a trivial JAX op run. This ensures runtime is consistent before loading MaxText.


In [4]:
# 4) TPU device and trivial op
import jax, jax.numpy as jnp
n = jax.device_count()
print(f"TPU devices: {n}")
print("Trivial JAX op:", jnp.add(1, 2))
if n != 8:
    print("⚠️ Warning: Expected 8 TPU cores. Verify accelerator is TPU v3-8.")


TPU devices: 8
Trivial JAX op: 3


## 5. Configure Kaggle Dataset Checkpoint Path
Set the path to the uploaded Kaggle dataset containing the MaxText Orbax checkpoint and validate key files exist.
- Accept either `<slug>/llama-3.1-8b-maxtext-checkpoint/` or directly `<slug>/` structures.


In [5]:
# 5) Determine checkpoint directory in Kaggle input
from pathlib import Path

DATASET_SLUG = "llama-3-1-8b-maxtext-checkpoint"  # change to your dataset slug if different
ROOT = Path("/kaggle/input") / DATASET_SLUG

inner = ROOT / "llama-3.1-8b-maxtext-checkpoint"
if (inner / "_CHECKPOINT_METADATA").exists() or (inner / "0").exists():
    base = inner
else:
    base = ROOT

# prefer step dir "0" if exists, else last numeric
step = None
if (base / "0").exists():
    step = base / "0"
else:
    nums = [p for p in base.iterdir() if p.is_dir() and p.name.isdigit()]
    if nums:
        step = sorted(nums, key=lambda p: int(p.name))[-1]

CKPT_DIR = step if step else base
print("Dataset root:", ROOT)
print("Checkpoint dir:", CKPT_DIR)

required = [
    CKPT_DIR / "_CHECKPOINT_METADATA",
    CKPT_DIR / "items" / "_METADATA",
]
for p in required:
    print("Exists", p, p.exists())

items_dir = CKPT_DIR / "items"
print("Items dir:", items_dir, items_dir.exists())


Dataset root: /kaggle/input/llama-3-1-8b-maxtext-checkpoint
Checkpoint dir: /kaggle/input/llama-3-1-8b-maxtext-checkpoint
Exists /kaggle/input/llama-3-1-8b-maxtext-checkpoint/_CHECKPOINT_METADATA True
Exists /kaggle/input/llama-3-1-8b-maxtext-checkpoint/items/_METADATA True
Items dir: /kaggle/input/llama-3-1-8b-maxtext-checkpoint/items True


## 6. Generate Minimal MaxText Config
Create a tiny YAML config with `load_parameters_path` pointing to the checkpoint and `steps: 1` for a minimal verification run.


In [6]:
# 6) Write minimal config YAML
import yaml
from pathlib import Path

CONFIG_DIR = Path("/kaggle/working/config")
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = CONFIG_DIR / "minimal_maxtext_config.yaml"

cfg = {
    "run_name": "llama31_8b_verify",
    "load_parameters_path": str(CKPT_DIR),
    "steps": 1,
    "dataset_type": "none",
}

with open(CONFIG_PATH, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Config:")
print(CONFIG_PATH.read_text())


Config:
run_name: llama31_8b_verify
load_parameters_path: /kaggle/input/llama-3-1-8b-maxtext-checkpoint
steps: 1
dataset_type: none



## 7. Verify MaxText Entrypoint and Run Minimal Step
Detect available entrypoints in this commit and run a minimal `steps=1` invocation using our config. Adjusts to available script names if needed.


In [7]:
# 7) Locate entrypoint and attempt minimal run
%cd /kaggle/working/maxtext

import os, subprocess

candidates = [
    "MaxText/train.py",
    "MaxText/train_compile.py",
    "MaxText/launch.py",
]

print("Entrypoint candidates:")
found = [c for c in candidates if os.path.exists(c)]
for c in candidates:
    print("-", c, "FOUND" if c in found else "missing")

entry = None
for c in candidates:
    if c in found:
        entry = c
        break

if entry:
    print("Using entrypoint:", entry)
    try:
        help_out = subprocess.run(['python3', entry, '--help'], capture_output=True, text=True, timeout=30)
        print("Help (first 800 chars):\n", help_out.stdout[:800])
    except Exception as e:
        print("Help failed:", e)

    # Minimal run
    cmd = ['python3', entry, f'--config={CONFIG_PATH}']
    print("\nRunning:", " ".join(cmd))
    try:
        res = subprocess.run(cmd, capture_output=True, text=True, timeout=240)
        print("Return code:", res.returncode)
        print("STDOUT (first 1200):\n", res.stdout[:1200])
        if res.stderr:
            print("STDERR (first 800):\n", res.stderr[:800])
    except subprocess.TimeoutExpired:
        print("⏰ TIMEOUT: verification run exceeded 240s")
else:
    print("No known entrypoints found in this commit. Inspect repository structure.")


/usr/local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/kaggle/working/maxtext
Entrypoint candidates:
- MaxText/train.py FOUND
- MaxText/train_compile.py FOUND
- MaxText/launch.py missing
Using entrypoint: MaxText/train.py
Help failed: Command '['python3', 'MaxText/train.py', '--help']' timed out after 30 seconds

Running: python3 MaxText/train.py --config=/kaggle/working/config/minimal_maxtext_config.yaml
Return code: 1
STDOUT (first 1200):
 
STDERR (first 800):
 Traceback (most recent call last):
  File "/kaggle/working/maxtext/MaxText/train.py", line 50, in <module>
    from layers import models
  File "/kaggle/working/maxtext/MaxText/layers/models.py", line 27, in <module>
    from layers import attentions
  File "/kaggle/working/maxtext/MaxText/layers/attentions.py", line 27, in <module>
    from jax.experimental.pallas.ops import attention as pallas_attention
ImportError: cannot import name 'attention' from 'jax.experimental.pallas.ops' (/usr/local/lib/python3.10/site-packages/jax/experimental/pallas/ops/__init__.py)

